In [0]:
# ==========================================================
# Delta tables DDL - bronze / silver / gold
# Run AFTER automotive_db.py (catalog + schema + volumes) and
# control_table.py (watermark table).
# ==========================================================

DDL_STATEMENTS = [
    # ---------- BRONZE (raw file landing zone tables) ----------
    """
    CREATE TABLE IF NOT EXISTS automotive_catalog.automotive_mfg.bronze_vehicle_production (
        vin_number STRING,
        plant_code STRING,
        assembly_line STRING,
        model_name STRING,
        production_stage STRING,
        production_status STRING,
        production_date DATE,
        shift STRING,
        op_code STRING,
        ingest_ts TIMESTAMP,
        source_file_name STRING,
        batch_run_id STRING,
        load_date DATE
    )
    USING DELTA
    """,
    """
    CREATE TABLE IF NOT EXISTS automotive_catalog.automotive_mfg.bronze_quality_inspection (
        inspection_id STRING,
        vin_number STRING,
        inspection_station STRING,
        defect_category STRING,
        defect_severity STRING,
        rework_required STRING,
        inspection_date DATE,
        op_code STRING,
        ingest_ts TIMESTAMP,
        source_file_name STRING,
        batch_run_id STRING,
        load_date DATE
    )
    USING DELTA
    """,
    """
    CREATE TABLE IF NOT EXISTS automotive_catalog.automotive_mfg.bronze_supplier_delivery (
        delivery_id STRING,
        supplier_id STRING,
        part_number STRING,
        part_description STRING,
        plant_code STRING,
        planned_qty INT,
        delivered_qty INT,
        delivery_status STRING,
        delivery_date DATE,
        op_code STRING,
        ingest_ts TIMESTAMP,
        source_file_name STRING,
        batch_run_id STRING,
        load_date DATE
    )
    USING DELTA
    """,
    # ---------- SILVER (cleaned, deduplicated) ----------
    """
    CREATE TABLE IF NOT EXISTS automotive_catalog.automotive_mfg.silver_vehicle_production (
        vin_number STRING,
        plant_code STRING,
        assembly_line STRING,
        model_name STRING,
        production_stage STRING,
        production_status STRING,
        production_date DATE,
        shift STRING,
        op_code STRING,
        processed_ts TIMESTAMP,
        batch_run_id STRING
    )
    USING DELTA
    """,
    """
    CREATE TABLE IF NOT EXISTS automotive_catalog.automotive_mfg.silver_quality_inspection (
        inspection_id STRING,
        vin_number STRING,
        inspection_station STRING,
        defect_category STRING,
        defect_severity STRING,
        rework_required STRING,
        inspection_date DATE,
        op_code STRING,
        processed_ts TIMESTAMP,
        batch_run_id STRING
    )
    USING DELTA
    """,
    """
    CREATE TABLE IF NOT EXISTS automotive_catalog.automotive_mfg.silver_supplier_delivery (
        delivery_id STRING,
        supplier_id STRING,
        part_number STRING,
        part_description STRING,
        plant_code STRING,
        planned_qty INT,
        delivered_qty INT,
        delivery_status STRING,
        delivery_date DATE,
        op_code STRING,
        processed_ts TIMESTAMP,
        batch_run_id STRING
    )
    USING DELTA
    """,
    # ---------- GOLD (SCD2 KPI table) ----------
    """
    CREATE TABLE IF NOT EXISTS automotive_catalog.automotive_mfg.gold_vehicle_manufacturing_kpi (
        surrogate_key BIGINT GENERATED ALWAYS AS IDENTITY,
        vin_number STRING,
        plant_code STRING,
        assembly_line STRING,
        model_name STRING,
        production_stage STRING,
        production_status STRING,
        total_defects INT,
        high_severity_defects INT,
        rework_required_flag STRING,
        supplier_delay_flag STRING,
        planned_parts_qty INT,
        delivered_parts_qty INT,
        delivery_fill_rate DOUBLE,
        quality_risk_level STRING,
        op_code STRING,
        is_current BOOLEAN,
        effective_from TIMESTAMP,
        effective_to TIMESTAMP,
        created_ts TIMESTAMP,
        updated_ts TIMESTAMP
    )
    USING DELTA
    """,
]

for ddl in DDL_STATEMENTS:
    spark.sql(ddl)

print(f"[SETUP] {len(DDL_STATEMENTS)} Delta tables ensured")